# Lab 18 — Responsible Model Validation
Calibration, threshold selection, bootstrap uncertainty and distribution shift. Streamlined Colab edition.

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV,calibration_curve
from sklearn.metrics import roc_auc_score,brier_score_loss,confusion_matrix,recall_score,f1_score
X,y=load_breast_cancer(return_X_y=True); y=(y==0).astype(int); Xa,Xte,ya,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y); Xtr,Xv,ytr,yv=train_test_split(Xa,ya,test_size=.2,random_state=42,stratify=ya); sc=StandardScaler(); Xtrs=sc.fit_transform(Xtr); Xvs=sc.transform(Xv); Xtes=sc.transform(Xte)

In [ ]:
base=LogisticRegression(max_iter=3000,class_weight='balanced'); cal=CalibratedClassifierCV(base,method='sigmoid',cv=5); cal.fit(Xtrs,ytr); pv=cal.predict_proba(Xvs)[:,1]; pt=cal.predict_proba(Xtes)[:,1]; print('Test ROC-AUC',roc_auc_score(yte,pt)); print('Brier',brier_score_loss(yte,pt))

In [ ]:
def stat(y,p,t):
 pred=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred).ravel(); return recall_score(y,pred),tn/(tn+fp),f1_score(y,pred)
rows=[]
for t in np.arange(.05,.96,.01):
 se,sp,f1=stat(yv,pv,t); rows.append([t,se,sp,f1,se+sp-1])
tab=pd.DataFrame(rows,columns=['Threshold','Sensitivity','Specificity','F1','Youden']); t=float(tab.loc[tab.Youden.idxmax(),'Threshold']); print('Selected threshold',t); print('Test sensitivity/specificity/F1',stat(yte,pt,t))

In [ ]:
rng=np.random.default_rng(42); vals=[]
for _ in range(1000):
 i=rng.integers(0,len(yte),len(yte)); vals.append(roc_auc_score(yte[i],pt[i]) if len(np.unique(yte[i]))==2 else np.nan)
vals=np.array(vals); vals=vals[~np.isnan(vals)]; print('ROC-AUC 95% bootstrap CI',np.percentile(vals,[2.5,97.5]))
shift=Xtes+rng.normal(0,.2,Xtes.shape); ps=cal.predict_proba(shift)[:,1]; print('Shifted ROC-AUC',roc_auc_score(yte,ps),'Shifted Brier',brier_score_loss(yte,ps))